<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L02-chat-templates-and-masking/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T04-L02-chat-templates-and-masking/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T04-L02-chat-templates-and-masking/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T04-L02-chat-templates-and-masking/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T04-L02 · Chat templates and loss masking: five bugs that train a broken assistant, and the tests that catch them

**You will build:** the three pieces of a supervised fine-tuning pipeline that most silent
failures hide in (a chat-template renderer, an assistant-only loss mask, a padding-free packer),
then break each piece on purpose, fine-tune a small transformer on the broken data, measure what
each bug does to held-out behaviour, and write the five unit tests that catch them *before* a
single training step runs.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, no GPU, no download, far under 8 GiB

By the end you will be able to:
1. Implement a chat-template renderer, an assistant-only loss mask and a packer whose attention
   mask is block-diagonal and whose position ids restart at every sample.
2. Fine-tune one pretrained numpy transformer seven times (five planted template, masking,
   end-token or packing bugs, and the two clean runs they are compared with), and measure
   held-out exact match, first-token accuracy and stop rate for every run.
3. Measure which bugs leave the training loss looking healthy while held-out behaviour breaks.
4. Write five pre-training tests, one per bug, each returning True on the buggy pipeline and
   False on a correct one, including a packing-invariance test that compares a sample's logits
   packed against alone.

**No PyTorch, no `transformers`, no TRL.** The model is the numpy transformer block you met in
`T03-L03-nanolm-cpu-slice`, copied in and extended; the special tokens follow the reserved-id idea
of `T03-L01-bpe-from-scratch`. Everything here is the mechanism that `apply_chat_template`,
`assistant_only_loss` and `packing=True` hide, small enough to read in one sitting.

**About the `if __name__ == "__main__":` guards.** They let the autograder import this file
without running the fine-tunes. A Jupyter kernel sets `__name__` to `"__main__"`, so every
guarded cell still runs when you execute the notebook top to bottom.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L01`, `T03-L03`

**This lesson assumes you can already:**

- write Python functions, loops and list comprehensions, and index, slice and mask numpy arrays
- say what next-token prediction trains: inputs, targets shifted left by one, and a loss averaged over positions (T03-L03 builds it)
- explain what a token and a special token are, and why a tokeniser reserves ids that text cannot produce (T03-L01 builds a tokeniser)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Chat templates (Transformers documentation)](https://huggingface.co/docs/transformers/chat_templating) · Hugging Face · *free* — how apply_chat_template turns messages into tokens, what add_generation_prompt does and the special-token pitfalls the lesson plants in miniature
- [Chat Templates (LLM Course, chapter 11)](https://huggingface.co/learn/llm-course/chapter11/2) · Hugging Face · *free* — a gentler walk through the same ground, with the supervised fine-tuning chapter that follows it
- [The Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/) · Harvard NLP (Rush et al.) · *free* — the attention block and its mask, line by line, which is what the lesson's model and its block-diagonal packing mask are built from

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed by the code itself.
import contextlib
import io
import itertools
import os
import signal
import sys
import threading
import time
from typing import Callable, NamedTuple

import numpy as np

print("python", sys.version.split()[0])
print("numpy ", np.__version__)

# Every check and every demo below runs through `_try`, so pressing Run all before you have
# written a line reaches the progress board at the foot of the notebook instead of stopping at
# the first stub.
# The eight exercises, in the order the notebook meets them: (label, function, where).
_EXERCISES = [("exercise 1", "render_chat", "section 5"),
              ("exercise 2", "build_example", "section 6"),
              ("exercise 3", "pack", "section 8"),
              ("exercise 4", "detects_template_drift", "section 11"),
              ("exercise 5", "detects_prompt_in_loss", "section 12"),
              ("exercise 6", "detects_unlearned_stop", "section 13"),
              ("exercise 7", "detects_shifted_mask", "section 14"),
              ("exercise 8", "detects_cross_sample_leak", "section 15")]
_FUNCTION = {label: function for label, function, _ in _EXERCISES}
_FOR_PIPELINE = ("exercise 1", "exercise 2", "exercise 3")   # the pipeline the bugs are planted in
_STATUS: dict = {}       # label -> "passed" | "failed" | "not started", for the progress board
_WAITING_ON: dict = {}   # label -> the stub whose NotImplementedError stopped it


def _unfinished(exc: BaseException) -> str:
    """The function that raised NotImplementedError: the innermost frame of the traceback.

    Exercise 4 calls the pipeline built from exercises 1 to 3, so an unfilled `render_chat` stops
    exercise 4's check too, and naming the stub that actually raised says which one to go back to.
    """
    tb = exc.__traceback__
    while tb is not None and tb.tb_next is not None:
        tb = tb.tb_next
    return tb.tb_frame.f_code.co_name if tb is not None else ""


def _named(labels: list) -> str:
    """["exercise 3"] -> "exercise 3 (`pack`)"; several -> "exercises 1, 2 and 3"."""
    if len(labels) == 1:
        return f"{labels[0]} (`{_FUNCTION[labels[0]]}`)"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so, by name. A wrong answer prints the
    check's own message, which names the likely mistake, and the notebook carries on, so one
    broken exercise never hides the feedback on the others. A demo names the exercises it
    `needs`: until each has passed its check, the demo says which one it is waiting for and
    skips, so it never runs on an answer a check has just rejected. Nothing is swallowed:
    every outcome is recorded for the progress board, and a script run exits non-zero there if
    any check came back wrong. A check still running after `_CHECK_SECONDS` (a demo:
    `_DEMO_SECONDS`) is stopped and counts as failed, so a loop that never ends cannot hang the
    notebook.
    """
    waiting = [name for name, _, _ in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped, needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _FUNCTION else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        stub = _unfinished(exc)
        _STATUS[label], _WAITING_ON[label] = "not started", stub
        owner = [name for name, function in _FUNCTION.items() if function == stub]
        if owner and owner[0] != label:
            print(f"{label}: skipped, needs {_named(owner)} first; it still raises "
                  "NotImplementedError.")
        else:
            todo = f"`{stub}`" if stub else "a stub"
            print(f"{label}: not implemented yet, {todo} still raises NotImplementedError. "
                  "Fill it in, then re-run this cell.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        print(f"{label}: FAILED, {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


# A check that never returns would freeze the notebook, and a loop that appends as it spins can
# eat gigabytes in seconds. Every exercise check runs under a time limit far above what a right
# answer needs (each takes milliseconds); a demo that trains gets a longer one.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 60


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str,
                why: str = "a loop that never reaches its stopping condition never ends",
                memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit, or an enclosing guard) keeps its deadline if it is due first and is put back,
    with the time it had left, afterwards. A correct answer finishes far inside the limit, and
    nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = f"{what} was still running after {seconds:g} s: {why}"
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


# Everything runs in float64: the finite-difference check below is meaningless in float32.
DTYPE = np.float64

## 1. A chat vocabulary: characters, plus five ids that text can never produce

A chat model does not see "messages". It sees one flat sequence of token ids in which a few
**special tokens** mark who is speaking and where a turn ends. A tokeniser reserves those ids
*above* the learned vocabulary, so ordinary text can never produce them: `T03-L01` does the same
with its `BOUNDARY` symbol, which sits one past the byte values and which no merge ever
crosses. This lesson keeps the idea and shrinks everything else: a small text vocabulary
(lowercase, a space, a full stop, a newline, and the capitals A to H) and five specials.

| id | special | job |
|---|---|---|
| `SYSTEM`, `USER`, `ASSISTANT` | role markers | open a turn and say whose it is |
| `END` | end of turn | closes every turn; on the assistant's turn it is also the model's *stop signal* |
| `PAD` | padding | fills short examples up to a common length; carries no meaning |

The cell builds the vocabulary and a helper that prints ids readably. Run it.

In [ ]:
TEXT_CHARS = "abcdefghijklmnopqrstuvwxyz .\nABCDEFGH"
SPECIAL_TOKENS = ("<|system|>", "<|user|>", "<|assistant|>", "<|end|>", "<|pad|>")
TEXT_ID = {c: i for i, c in enumerate(TEXT_CHARS)}
SPECIAL_ID = {s: len(TEXT_CHARS) + i for i, s in enumerate(SPECIAL_TOKENS)}
VOCAB_SIZE = len(TEXT_CHARS) + len(SPECIAL_TOKENS)
SYSTEM, USER, ASSISTANT, END, PAD = (SPECIAL_ID[s] for s in SPECIAL_TOKENS)
ROLE_TOKEN = {"system": SYSTEM, "user": USER, "assistant": ASSISTANT}


def encode_text(text: str) -> list:
    """Plain text to ids, one id per character. A character outside the vocabulary is a KeyError."""
    return [TEXT_ID[c] for c in text]


def pretty(ids) -> str:
    """Ids back to a readable string: specials in angle brackets, newlines shown as \\n."""
    out = []
    for i in ids:
        i = int(i)
        out.append(SPECIAL_TOKENS[i - len(TEXT_CHARS)] if i >= len(TEXT_CHARS)
                   else TEXT_CHARS[i].replace("\n", "\\n"))
    return "".join(out)


print(f"{len(TEXT_CHARS)} text characters + {len(SPECIAL_TOKENS)} specials = {VOCAB_SIZE} ids")
print(f"specials start at id {min(SPECIAL_ID.values())}; the last text character is id "
      f"{len(TEXT_CHARS) - 1}")
_demo = [USER] + encode_text("abcd") + [END, ASSISTANT] + encode_text("ABCD") + [END]
print(f"a conversation as ids:    {_demo}")
print(f"the same, made readable:  {pretty(_demo)}")

## 2. What real chat templates look like, and why you cannot trust one by eye

Every instruction-tuned model family fixes its own markers, its own separators and its own
idea of what follows the assistant header. As of 2 October 2026, from each family's own
published template or documentation (sources and quotes in `claims.yaml`):

- **Qwen3.5** (ChatML style): `<|im_start|>role\ncontent<|im_end|>\n`. Its template asks for the
  reply by appending `<|im_start|>assistant\n`, and unless thinking is explicitly enabled it adds
  an *empty* `<think>\n\n</think>\n\n` block after that header, so the served prompt ends with
  tokens that an assistant turn from the earlier history is not rendered with.
- **SmolLM3**: the same ChatML turn markers, with each assistant turn wrapped in `{% generation %}`
  and `{% endgeneration %}` keywords, which is the hook an assistant-only loss mask reads.
- **Llama 3**: `<|start_header_id|>role<|end_header_id|>` then the text then `<|eot_id|>`; the
  model "expects the assistant header at the end of the prompt to start completing it".
- **Gemma**: `<start_of_turn>role\ncontent<end_of_turn>\n`, with the prompt prefix for a reply
  being `<start_of_turn>model\n`.

Four families, three turn formats, and different rules for what a served prompt ends with. A pipeline that renders training data with one convention and
serves with another, or masks the loss using a template it did not render with, fails silently:
nothing raises, the loss falls, and the model is simply wrong when someone talks to it. Hugging
Face's own `apply_chat_template` documentation describes its `add_generation_prompt` argument as
adding "tokens to the end of the chat that indicate the start of an assistant response", and
TRL's `assistant_only_loss` documentation says the template must contain the generation keywords
above. Those are the mechanisms this lesson rebuilds in miniature.

The toy template used here has one rule: **every message is its role marker, its text, then
`END`**, and the reply is requested by appending `ASSISTANT`. Exercise 1 writes it.

In [ ]:
# The same user message asked for a reply, formatted by two of those conventions (the ChatML turn
# format of Qwen3.5 and SmolLM3, and Gemma's), written out as plain strings. As of 2 October 2026;
# the lines they implement are quoted in `claims.yaml`.
def chatml(messages: list, add_generation_prompt: bool = False) -> str:
    text = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return text + ("<|im_start|>assistant\n" if add_generation_prompt else "")


def gemma(messages: list, add_generation_prompt: bool = False) -> str:
    role = {"user": "user", "assistant": "model"}
    text = "".join(f"<start_of_turn>{role[m['role']]}\n{m['content']}<end_of_turn>\n"
                   for m in messages)
    return text + ("<start_of_turn>model\n" if add_generation_prompt else "")


_ask = [{"role": "user", "content": "hi"}]
_a, _b = chatml(_ask, True), gemma(_ask, True)
print(f"ChatML prompt: {_a!r}")
print(f"Gemma prompt:  {_b!r}")
print(f"{len(_a)} and {len(_b)} characters for the same {len(_ask[0]['content'])} letters of user "
      f"text, and the two "
      f"prompts share {len(os.path.commonprefix([_a, _b]))} leading character(s)")

## 3. The model, copied from T03-L03 and extended

The numpy transformer block of `T03-L03-nanolm-cpu-slice` is copied in below: token and position
embeddings, one causal attention head, a residual connection, a linear read-out, a hand-written
backward pass and Adam. A lesson cannot import another lesson's files, so it carries its own
copy, 32 wide instead of 48 and with 36 positions instead of 32. Beyond those sizes, three
things are different, and each one is something a chat pipeline needs:

- **The attention mask is an input**, shape `(batch, T, T)`, not an implied lower triangle. A
  packed row holds several conversations and each must only see itself.
- **Position ids are an input.** A packed conversation must count positions from zero again.
- **The loss takes per-position weights.** A weight of 0 means "do not learn from this token",
  which is all that loss masking is.

The backward pass is given, and the next cell proves it against finite differences with all
three changes active, the way `T03-L03` proved the original.

In [ ]:
D_MODEL = 32         # width of the residual stream
CONTEXT = 36         # positions the model has an embedding for
INIT_SCALE = 0.1     # standard deviation of the initial weights


def softmax(x: np.ndarray) -> np.ndarray:
    """Softmax over the last axis, shifted by the row maximum so it cannot overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def attention_forward(x, wq, wk, wv, allowed):
    """One attention head. `allowed[b, i, j]` says whether position i may look at position j."""
    d = x.shape[-1]
    q, k, v = x @ wq, x @ wk, x @ wv
    scale = 1.0 / np.sqrt(d)
    scores = np.where(allowed, (q @ k.transpose(0, 2, 1)) * scale, -np.inf)
    a = softmax(scores)
    return a @ v, {"X": x, "Q": q, "K": k, "V": v, "A": a, "allowed": allowed, "scale": scale,
                   "Wq": wq, "Wk": wk, "Wv": wv}


def attention_backward(d_context, cache):
    """Backward pass of `attention_forward`: (dX, dWq, dWk, dWv)."""
    x, q, k, v, a = cache["X"], cache["Q"], cache["K"], cache["V"], cache["A"]
    d = x.shape[-1]
    d_a = d_context @ v.transpose(0, 2, 1)
    d_v = a.transpose(0, 2, 1) @ d_context
    d_scores = a * (d_a - (d_a * a).sum(axis=-1, keepdims=True))
    d_scores = np.where(cache["allowed"], d_scores, 0.0) * cache["scale"]
    d_q, d_k = d_scores @ k, d_scores.transpose(0, 2, 1) @ q
    flat = lambda t: t.reshape(-1, d)
    d_x = d_q @ cache["Wq"].T + d_k @ cache["Wk"].T + d_v @ cache["Wv"].T
    return d_x, flat(x).T @ flat(d_q), flat(x).T @ flat(d_k), flat(x).T @ flat(d_v)


def init_params(seed: int = 0, scale: float = INIT_SCALE) -> dict:
    """Fresh parameters, float64, from a seeded generator."""
    rng = np.random.default_rng(seed)
    params = {"E": rng.normal(0, scale, (VOCAB_SIZE, D_MODEL)),
              "P": rng.normal(0, scale, (CONTEXT, D_MODEL))}
    for name in ("Wq", "Wk", "Wv", "Wo"):
        params[name] = rng.normal(0, scale, (D_MODEL, D_MODEL))
    params["W1"] = rng.normal(0, scale, (D_MODEL, VOCAB_SIZE))
    params["b1"] = np.zeros(VOCAB_SIZE, dtype=DTYPE)
    return params


def forward(params, ids, pos, allowed):
    """ids, pos: (B, T) int arrays; allowed: (B, T, T) bool. Returns (logits (B, T, V), cache)."""
    x = params["E"][ids] + params["P"][pos]
    context, att = attention_forward(x, params["Wq"], params["Wk"], params["Wv"], allowed)
    h = x + context @ params["Wo"]
    return h @ params["W1"] + params["b1"], {"ids": ids, "pos": pos, "ctx": context, "att": att,
                                              "H": h}


def masked_loss(logits, targets, weights):
    """Weighted mean negative log-likelihood in nats: sum(w * nll) / sum(w). Returns (loss, nll)."""
    m = logits.max(axis=-1, keepdims=True)
    lse = m[..., 0] + np.log(np.exp(logits - m).sum(axis=-1))
    nll = lse - np.take_along_axis(logits, targets[..., None], axis=-1)[..., 0]
    return float((nll * weights).sum() / max(float(weights.sum()), 1.0)), nll


def backward(params, cache, logits, targets, weights):
    """Gradient of `masked_loss` with respect to every parameter."""
    d = D_MODEL
    d_logits = softmax(logits)
    chosen = np.take_along_axis(d_logits, targets[..., None], axis=-1)
    np.put_along_axis(d_logits, targets[..., None], chosen - 1.0, axis=-1)
    d_logits = d_logits * (weights[..., None] / max(float(weights.sum()), 1.0))
    h = cache["H"]
    grads = {"W1": h.reshape(-1, d).T @ d_logits.reshape(-1, VOCAB_SIZE),
             "b1": d_logits.sum(axis=(0, 1))}
    d_h = d_logits @ params["W1"].T
    grads["Wo"] = cache["ctx"].reshape(-1, d).T @ d_h.reshape(-1, d)
    d_x_att, grads["Wq"], grads["Wk"], grads["Wv"] = attention_backward(d_h @ params["Wo"].T,
                                                                        cache["att"])
    d_x = d_h + d_x_att
    grads["P"] = np.zeros_like(params["P"])
    np.add.at(grads["P"], cache["pos"], d_x)
    grads["E"] = np.zeros_like(params["E"])
    np.add.at(grads["E"], cache["ids"], d_x)
    return grads


def adam_init(params: dict) -> dict:
    return {k: [np.zeros_like(v), np.zeros_like(v), 0] for k, v in params.items()}


def adam_step(params, grads, state, lr=3e-3, beta1=0.9, beta2=0.999, eps=1e-8) -> None:
    """One Adam update, in place, bias-corrected."""
    for k, p in params.items():
        m, v, _ = state[k]
        state[k][2] += 1
        t = state[k][2]
        m *= beta1
        m += (1 - beta1) * grads[k]
        v *= beta2
        v += (1 - beta2) * grads[k] ** 2
        p -= lr * (m / (1 - beta1 ** t)) / (np.sqrt(v / (1 - beta2 ** t)) + eps)

In [ ]:
def gradient_check(seed: int = 900, scale: float = 0.2, probes: int = 6) -> float:
    """Finite-difference every parameter with a random attention mask, random loss weights and
    random position ids: the three things this copy changed. Returns the worst error as a
    fraction of the tolerance (anything above 1.0 is a real disagreement)."""
    rng = np.random.default_rng(seed)
    params = init_params(seed, scale)
    params["b1"] = rng.normal(0, 0.02, VOCAB_SIZE)
    n, t = 3, 9
    ids = rng.integers(0, VOCAB_SIZE, (n, t))
    targets = rng.integers(0, VOCAB_SIZE, (n, t))
    weights = (rng.random((n, t)) < 0.6).astype(DTYPE)
    pos = rng.integers(0, CONTEXT, (n, t))
    allowed = np.tril(rng.random((n, t, t)) < 0.7) | np.eye(t, dtype=bool)

    def loss_of() -> float:
        logits, _ = forward(params, ids, pos, allowed)
        return masked_loss(logits, targets, weights)[0]

    logits, cache = forward(params, ids, pos, allowed)
    grads = backward(params, cache, logits, targets, weights)
    worst = 0.0
    for name, array in params.items():
        flat = array.reshape(-1)
        for q in rng.integers(0, flat.size, size=probes):
            old = flat[q]
            flat[q] = old + 1e-5
            up = loss_of()
            flat[q] = old - 1e-5
            down = loss_of()
            flat[q] = old
            numeric, analytic = (up - down) / 2e-5, float(grads[name].reshape(-1)[q])
            tolerance = 1e-9 + 1e-5 * max(abs(numeric), abs(analytic))
            worst = max(worst, abs(numeric - analytic) / tolerance)
    return worst


_worst = gradient_check()
print(f"parameters: {sum(v.size for v in init_params().values())}")
# The worst error itself is rounding noise, which changes with the BLAS library underneath numpy,
# so the cell prints the verdict, not the ratio.
print(f"gradient check, all {len(init_params())} parameter blocks, random mask + weights + "
      f"positions: every probed derivative within its tolerance: {_worst <= 1.0}")
assert _worst <= 1.0, "the copied backward pass disagrees with finite differences"

## 4. Pretraining a small base model

Fine-tuning starts from a model that has already learned *something*. This cell pretrains a
base checkpoint on plain text generated from a tiny word grammar (a fixed seed, no download).
The base has never seen a capital letter or any of the five special tokens, which is exactly
the situation of a real base model meeting a new chat template: the embeddings for the new
tokens start as noise, and fine-tuning has to teach them everything.

In [ ]:
PRETRAIN_WORDS = ["the", "cat", "sat", "on", "a", "mat", "dog", "ran", "to", "red", "big", "sun",
                  "and", "moon", "fell", "up", "down", "in", "one", "two"]
PRETRAIN_STEPS = 400


def pretrain_base(seed: int = 0, steps: int = PRETRAIN_STEPS):
    """Train the base checkpoint. Returns (params, first_loss, last_loss, ids_seen)."""
    rng = np.random.default_rng(seed)
    text = "".join(" ".join(rng.choice(PRETRAIN_WORDS, 40)) + ".\n" for _ in range(60))
    data = np.array(encode_text(text))
    params, state = init_params(seed), None
    state = adam_init(params)
    batch, window = 16, 32
    pos = np.tile(np.arange(window), (batch, 1))
    allowed = np.broadcast_to(np.tril(np.ones((window, window), dtype=bool)), (batch, window, window))
    losses = []
    for _ in range(steps):
        starts = rng.integers(0, len(data) - window - 1, batch)
        x = np.stack([data[s:s + window] for s in starts])
        y = np.stack([data[s + 1:s + window + 1] for s in starts])
        logits, cache = forward(params, x, pos, allowed)
        losses.append(masked_loss(logits, y, np.ones((batch, window)))[0])
        adam_step(params, backward(params, cache, logits, y, np.ones((batch, window))), state)
    return params, float(np.mean(losses[:10])), float(np.mean(losses[-10:])), set(data.tolist())


BASE, _first, _last, _seen = pretrain_base()
print(f"pretraining: loss {_first:.3f} -> {_last:.3f} nats per character over {PRETRAIN_STEPS} steps "
      f"(a uniform guess scores {np.log(len(TEXT_CHARS)):.3f})")
_new = sorted(set(range(VOCAB_SIZE)) - _seen)
_n_special = sum(1 for i in _new if i >= len(TEXT_CHARS))
_capitals = "ABCDEFGH"
_n_capital = sum(1 for i in _new if i < len(TEXT_CHARS) and TEXT_CHARS[i] in _capitals)
print(f"the base model has seen {len(_seen)} of the {VOCAB_SIZE} ids; {len(_new)} are new to it: "
      f"{_n_special} of the {len(SPECIAL_TOKENS)} specials, {_n_capital} of the {len(_capitals)} "
      f"capitals and "
      f"{len(_new) - _n_special - _n_capital} other characters the grammar never uses")

## 5. Exercise 1 · `render_chat(messages, add_generation_prompt=False)`

The template itself. A conversation is a list of messages, each `{"role": ..., "content": ...}`.
Render it to one flat list of token ids by the one rule: **role marker, text, `END`**, for every
message in order. When `add_generation_prompt` is true, append the `ASSISTANT` marker last: that
is the "your turn" signal a server sends, and it must be the only difference.

<details><summary>💡 Hint 1 — what to think about</summary>

Say the rule out loud for a conversation with a system message, a user message and an assistant
message, and write down the ids you expect. Which token opens a turn? Which closes it, and does
the closing token depend on the role? Then ask what a role the template has never heard of
should do: silently skip the message, or refuse.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the messages in order. For each, look the role up in `ROLE_TOKEN` (raising `ValueError` for
an unknown role), then add the role id, then the ids of the text from `encode_text`, then `END`.
After the loop, add `ASSISTANT` only when asked. Build a new list; never touch the messages you
were handed.

</details>

In [ ]:
def render_chat(messages: list, add_generation_prompt: bool = False) -> list:
    """Render a conversation to token ids: role marker, text, END, for every message. The text
    is encoded exactly as given: nothing stripped, nothing added.

    Args:
        messages: a list of {"role": "system" | "user" | "assistant", "content": str} dicts.
        add_generation_prompt: when True, append the ASSISTANT marker after the last message.

    Raise ValueError for any other role (roles are case-sensitive: "User" is not "user").
    Do not modify `messages`.

    Example:
        >>> pretty(render_chat([{"role": "user", "content": "ab"}]))
        '<|user|>ab<|end|>'
        >>> pretty(render_chat([{"role": "user", "content": "ab"}], add_generation_prompt=True))
        '<|user|>ab<|end|><|assistant|>'

    Returns:
        A list of Python ints.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_render_chat() -> None:
    user = {"role": "user", "content": "abc"}
    got = render_chat([user])
    want = [USER] + encode_text("abc") + [END]
    assert got == want, (f"one user message rendered as {pretty(got)}, expected {pretty(want)}: "
                         "every message is role marker, text, then END.")
    assert all(type(i) is int for i in got), (
        f"the ids must be plain Python ints, got {type(got[0]).__name__}")
    served = render_chat([user], add_generation_prompt=True)
    assert served == want + [ASSISTANT], (
        f"with add_generation_prompt=True the prompt is {pretty(served)}; it must be the "
        f"conversation followed by exactly one ASSISTANT marker: {pretty(want + [ASSISTANT])}.")
    assert render_chat([user], False) == want, "add_generation_prompt=False must add nothing"
    chat = [{"role": "system", "content": "ab"}, user, {"role": "assistant", "content": "ABC"}]
    got = render_chat(chat)
    want = ([SYSTEM] + encode_text("ab") + [END] + [USER] + encode_text("abc") + [END]
            + [ASSISTANT] + encode_text("ABC") + [END])
    assert got == want, (f"a three-message chat rendered as {pretty(got)}, expected {pretty(want)}: "
                         "the assistant message closes with END too, and the system role has its "
                         "own marker.")
    assert render_chat([]) == [], "an empty conversation renders to no tokens"
    for role in ("tool", "User"):
        try:
            render_chat([{"role": role, "content": "a"}])
        except ValueError:
            pass
        else:
            raise AssertionError(f"the role {role!r} must raise ValueError, not be skipped or "
                                 "guessed: the template knows system, user and assistant exactly")
    before = [dict(m) for m in chat]
    render_chat(chat, True)
    assert chat == before, "render_chat modified the messages it was given"
    print(f"exercise 1 looks right: {pretty(render_chat(chat, True))}")

In [ ]:
if __name__ == "__main__":
    _try("exercise 1", _check_render_chat)

## 6. Exercise 2 · `build_example(messages)`: the assistant-only loss mask

Training does not learn from every token. A user's words are *input*: the model must be able to
read them, but a model trained to produce them learns to write the user's side of the
conversation. Only what the assistant says, **including the `END` that closes its turn**, is
something the model should learn to emit. The mask is a list of booleans, one per token id:
`True` means "this token is a target the model must learn to produce".

So `build_example` returns `(ids, supervised)` of equal length. The ids are exactly what
`render_chat` would give. For each assistant message, its text tokens and its `END` are
supervised. Everything else is not: the `ASSISTANT` marker (the server supplies that, the model
never has to), system and user turns, and their `END` tokens.

<details><summary>💡 Hint 1 — what to think about</summary>

Draw the rendered ids of a two-message conversation and colour the tokens the model should
learn to produce. Which of the assistant turn's tokens are *given* to the model at serving time
(the prompt) and which must it *generate*? Check the boundaries: the first and last coloured
token, and what happens with a conversation that has two assistant turns.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Build the ids message by message, exactly as the renderer does, and extend a second list in
step: one entry for the role marker (never supervised), then one per text token and one for
`END`, all of them supervised only when the message's role is the assistant's. The two lists
must end up the same length.

</details>

In [ ]:
def build_example(messages: list) -> tuple:
    """Token ids of a conversation, and which of them the model must learn to produce.

    Returns (ids, supervised): two lists of the same length. `ids` equals render_chat(messages).
    `supervised[i]` is True exactly for the text tokens of each assistant message and the END
    that closes it. The role markers, and every system or user token, are False. Raise ValueError
    for an unknown role.

    Example:
        >>> ids, sup = build_example([{"role": "user", "content": "ab"},
        ...                           {"role": "assistant", "content": "AB"}])
        >>> pretty(ids)
        '<|user|>ab<|end|><|assistant|>AB<|end|>'
        >>> [int(s) for s in sup]
        [0, 0, 0, 0, 0, 1, 1, 1]
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _marks(sup) -> str:
    """The mask as a string: S where a token is learned, a dot where it is only read."""
    return "".join("S" if s else "." for s in sup)


def _check_build_example() -> None:
    chat = [{"role": "user", "content": "abc"}, {"role": "assistant", "content": "ABC"}]
    ids, sup = build_example(chat)
    assert ids == [USER] + encode_text("abc") + [END, ASSISTANT] + encode_text("ABC") + [END], (
        f"the ids are {pretty(ids)}; they must equal render_chat of the same messages")
    assert len(sup) == len(ids), f"{len(sup)} mask entries for {len(ids)} ids: one per token"
    want = "......SSSS"
    assert _marks(sup) == want, (
        f"the mask is {_marks(sup)} over {pretty(ids)}; it must mark only the reply text and its "
        f"END ({want}). The ASSISTANT marker and everything the user said are prompt, not target.")
    many = [{"role": "system", "content": "ab"}, {"role": "user", "content": "ba"},
            {"role": "assistant", "content": "BA"}, {"role": "user", "content": "cd"},
            {"role": "assistant", "content": "CD"}]
    ids, sup = build_example(many)
    want = "...." + "...." + ".SSS" + "...." + ".SSS"
    assert _marks(sup) == want, (
        f"a five-message chat gave the mask {_marks(sup)}, expected {want}: each assistant turn "
        "is learned, the system and user turns and every ASSISTANT marker are not.")
    n_assistant = sum(m["role"] == "assistant" for m in many)
    assert sum(1 for i, s in enumerate(sup) if s and ids[i] == END) == n_assistant, (
        "every assistant turn must have its END supervised, not only the last one")
    assert build_example([])[0] == [] and build_example([])[1] == [], (
        "an empty conversation has no tokens and an empty mask")
    try:
        build_example([{"role": "tool", "content": "a"}])
    except ValueError:
        pass
    else:
        raise AssertionError("an unknown role must raise ValueError")
    print(f"exercise 2 looks right: {pretty(ids)}")
    print(f"                        {_marks(sup)}   (S = learned, . = only read)")

In [ ]:
if __name__ == "__main__":
    _try("exercise 2", _check_build_example)

## 7. Batching: padding, and packing

The task the model is fine-tuned on is deliberately plain, so that any failure is the pipeline's
and not the task's: the user sends four letters from `a` to `h`, the assistant replies with the
same four in capitals. Most of the possible payloads are never trained on: the model learns from
a few hundred and is judged on payloads it never saw, so memorising cannot pass (the next cell
prints the counts).

Examples come in different lengths, and a batch is a rectangle, so a trainer has two choices:

- **Pad** every example to a common length with `PAD` and give the padding loss weight 0.
  Simple, and it wastes the padded positions. `collate` below does it, and is given.
- **Pack** several examples end to end into one fixed-length row, so no position is wasted.
  Then the row holds *several conversations*, and two things must be true. Attention must be
  **block-diagonal**: a token may look back only within its own conversation. And position ids
  must **restart at zero** for each conversation, because the model learned what position 0
  means from conversations that start there. `pack` is exercise 3.

Both batchers return the same `Batch`: `inputs`, `targets` (the inputs shifted left by one),
`weights` (the mask, shifted the same way), `pos`, `allowed` and, per row, the `spans` that
say where each conversation sits. Run the cell: it builds the data and shows `collate` on one
hand-written example, so you can see the shift before you write a packer.

In [ ]:
LETTERS = "abcdefgh"
PAYLOAD_LEN = 4
ALL_PAYLOADS = ["".join(t) for t in itertools.product(LETTERS, repeat=PAYLOAD_LEN)]
_order = np.random.default_rng(1).permutation(len(ALL_PAYLOADS))
TRAIN_PAYLOADS = [ALL_PAYLOADS[i] for i in _order[:300]]
HELD_PAYLOADS = [ALL_PAYLOADS[i] for i in _order[300:428]]
PAD_LEN = 13      # input length of a padded batch
PACK_LEN = 36     # input length of a packed row: one position id for every embedding the model has


def conversation(payload: str) -> list:
    """The task: the user sends lowercase letters, the assistant answers in capitals."""
    return [{"role": "user", "content": payload}, {"role": "assistant", "content": payload.upper()}]


class Batch(NamedTuple):
    """What every batcher returns, and what the training step and the model consume."""
    inputs: np.ndarray    # (R, T) int64 token ids fed to the model
    targets: np.ndarray   # (R, T) int64 the token each position must predict (inputs shifted left)
    weights: np.ndarray   # (R, T) float64 loss weight on each target: 1.0 learned, 0.0 ignored
    pos: np.ndarray       # (R, T) int64 position ids
    allowed: np.ndarray   # (R, T, T) bool: may position i attend to position j?
    spans: list           # per row, a list of (start, end) input ranges, one per conversation


class Pipeline(NamedTuple):
    """The four pieces of an SFT pipeline. A bug is one of them swapped for a broken version."""
    build_example: Callable   # messages -> (ids, supervised): what training sees
    serve_render: Callable    # (messages, add_generation_prompt) -> ids: what the server sends
    collate: Callable         # (examples, max_len) -> Batch: pad to a common length
    pack: Callable            # (examples, max_len) -> Batch: pack into full rows


def collate(examples: list, max_len: int, pad_id: int = PAD, shift_mask: bool = True) -> Batch:
    """Pad each (ids, supervised) example to max_len + 1 ids, shift by one, build the Batch."""
    n = len(examples)
    ids = np.full((n, max_len + 1), pad_id, dtype=np.int64)
    supervised = np.zeros((n, max_len + 1), dtype=DTYPE)
    for row, (example_ids, example_sup) in enumerate(examples):
        if len(example_ids) > max_len + 1:
            raise ValueError(f"an example of {len(example_ids)} ids does not fit max_len={max_len}")
        ids[row, :len(example_ids)] = example_ids
        supervised[row, :len(example_sup)] = np.asarray(example_sup, dtype=DTYPE)
    weights = supervised[:, 1:] if shift_mask else supervised[:, :-1]
    pos = np.tile(np.arange(max_len, dtype=np.int64), (n, 1))
    allowed = np.broadcast_to(np.tril(np.ones((max_len, max_len), dtype=bool)),
                              (n, max_len, max_len)).copy()
    spans = [[(0, len(example_ids) - 1)] for example_ids, _ in examples]
    return Batch(ids[:, :-1], ids[:, 1:], weights, pos, allowed, spans)


def make_pipeline() -> Pipeline:
    """The pipeline built from YOUR exercises 1 to 3 and the given `collate`."""
    return Pipeline(build_example, render_chat, collate, pack)


if __name__ == "__main__":
    _overlap = len(set(TRAIN_PAYLOADS) & set(HELD_PAYLOADS))
    print(f"{len(ALL_PAYLOADS)} possible payloads: {len(TRAIN_PAYLOADS)} to train on, "
          f"{len(HELD_PAYLOADS)} held out, {_overlap} in both")
    _ids = [USER] + encode_text("ab") + [END, ASSISTANT] + encode_text("AB") + [END]
    _sup = [False] * 5 + [False, True, True, True]
    _demo_batch = collate([(_ids, _sup)], 11)
    print("one hand-written example, padded and shifted:")
    print("  inputs   ", pretty(_demo_batch.inputs[0]))
    print("  targets  ", pretty(_demo_batch.targets[0]))
    print("  weights  ", "".join(str(int(w)) for w in _demo_batch.weights[0]))
    print("each weight sits on the TARGET it scores: the first 1 is the position whose job is to "
          "predict the first reply letter")

## 8. Exercise 3 · `pack(examples, max_len)`

Pack `(ids, supervised)` examples, in the order given, into rows of exactly `max_len` input
positions. Each example contributes `len(ids) - 1` positions (its inputs `ids[:-1]`, its
targets `ids[1:]`, its weights `supervised[1:]` as floats). Fill a row until the next example
would not fit, then start a new row; an example that does not fit alone raises `ValueError`.

A row is padded to the right with `PAD` inputs and targets, weight 0, position 0. The mask is
the whole exercise:

- a real position may attend to an earlier-or-equal position **of its own example only**;
- a padding position may attend only to itself (an all-masked row would make the softmax `nan`).

`spans[r]` lists `(start, end)` for each example in row `r`: its inputs are `inputs[r, start:end]`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two arrays are enough to describe the mask: for every position, which example it belongs to
(a "segment id", with something distinct for padding) and where it sits in the row. When may
position i look at position j? Both conditions have to hold, and neither is enough alone.
Then ask what the position ids of the second example in a row should be.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

First split the examples into rows greedily, counting `len(ids) - 1` per example. Then fill
the arrays row by row, writing each example's inputs, targets, weights and position ids
(counting from zero again) at its offset and recording its segment. Build the mask for all
rows at once by comparing every position's segment with every other's and requiring the
column not to be later than the row, then let padding positions see themselves.

</details>

In [ ]:
def pack(examples: list, max_len: int) -> Batch:
    """Pack (ids, supervised) examples into rows of max_len input positions, with a
    block-diagonal causal mask and position ids that restart for every example.

    Args:
        examples: list of (ids, supervised) pairs, as `build_example` returns them.
        max_len: the number of input positions in a row.

    Rules:
        * Example k contributes inputs ids[:-1], targets ids[1:], weights supervised[1:] (floats)
          and position ids 0, 1, 2, ... It occupies len(ids) - 1 consecutive positions.
        * Rows are filled greedily in the order given: start a new row as soon as the next
          example would not fit in the space left. An example that does not fit even in an
          empty row raises ValueError.
        * Unused positions: inputs PAD, targets PAD, weights 0.0, position id 0, and they may
          attend only to themselves.
        * A real position i may attend to j only if j <= i and j belongs to the same example.
        * spans[r] is [(start, end), ...], one per example in row r, in order.

    Example (PAD is the padding id; two examples of 3 input positions each, one row of 8):
        >>> a = ([5, 6, 7, 8], [False, False, True, True])
        >>> b = ([9, 10, 11, 12], [False, False, True, True])
        >>> batch = pack([a, b], 8)
        >>> batch.inputs.tolist() == [[5, 6, 7, 9, 10, 11, PAD, PAD]]
        True
        >>> batch.pos.tolist()
        [[0, 1, 2, 0, 1, 2, 0, 0]]
        >>> batch.weights.tolist()
        [[0.0, 1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0]]
        >>> batch.allowed[0, 3].astype(int).tolist()      # b's first token sees only itself
        [0, 0, 0, 1, 0, 0, 0, 0]
        >>> batch.spans
        [[(0, 3), (3, 6)]]

    Returns:
        A Batch with inputs, targets, pos int64; weights float64; allowed bool (R, max_len, max_len).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_pack() -> None:
    def pack_(examples, max_len):
        with _time_limit(1.0, f"pack of {len(examples)} small example(s) into rows of {max_len}",
                         "a loop that starts a new row without placing an example never ends"):
            return pack(examples, max_len)

    a = ([5, 6, 7, 8], [False, False, True, True])
    b = ([9, 10, 11, 12, 13], [False, False, True, True, True])
    batch = pack_([a, b], 8)
    assert batch.inputs.shape == (1, 8) and batch.allowed.shape == (1, 8, 8), (
        f"two examples needing 3 + 4 = 7 positions should share one row of 8, got inputs "
        f"{batch.inputs.shape}")
    assert batch.inputs[0].tolist() == [5, 6, 7, 9, 10, 11, 12, PAD], (
        f"inputs are {batch.inputs[0].tolist()}: each example contributes ids[:-1], the last "
        "id of an example is only ever a target")
    assert batch.targets[0].tolist() == [6, 7, 8, 10, 11, 12, 13, PAD], (
        f"targets are {batch.targets[0].tolist()}: each example contributes ids[1:], so no "
        "position predicts the first token of the NEXT example")
    assert batch.weights[0].tolist() == [0, 1, 1, 0, 1, 1, 1, 0], (
        f"weights are {batch.weights[0].tolist()}: they are the mask shifted like the targets, "
        "supervised[1:] for each example, and 0 on padding")
    assert batch.pos[0].tolist() == [0, 1, 2, 0, 1, 2, 3, 0], (
        f"position ids are {batch.pos[0].tolist()}: they must restart at 0 for every example")
    assert batch.spans == [[(0, 3), (3, 7)]], f"spans are {batch.spans}"
    seg = [0, 0, 0, 1, 1, 1, 1]
    for i in range(7):
        for j in range(8):
            want = j < 7 and seg[j] == seg[i] and j <= i
            assert bool(batch.allowed[0, i, j]) == want, (
                f"position {i} may{'' if want else ' NOT'} attend to position {j}: attention "
                "must be causal AND stay inside the example, so the second example never sees "
                "the first")
    assert batch.allowed[0, 7].astype(int).tolist() == [0, 0, 0, 0, 0, 0, 0, 1], (
        "a padding position must attend only to itself: an all-masked row makes the softmax nan")
    assert batch.inputs.dtype == np.int64 and batch.weights.dtype == np.float64 \
        and batch.allowed.dtype == bool, (
            f"dtypes are {batch.inputs.dtype}, {batch.weights.dtype}, {batch.allowed.dtype}; "
            "expected int64, float64, bool")
    exact = pack_([a, a], 6)
    assert exact.inputs.shape == (1, 6), (
        f"two 3-position examples exactly fill a row of 6, but you made {exact.inputs.shape[0]} "
        "rows: only start a new row when the next example does NOT fit")
    split = pack_([a, b], 6)
    assert split.inputs.shape == (2, 6) and split.spans == [[(0, 3)], [(0, 4)]], (
        f"3 + 4 positions do not fit one row of 6; expected two rows, got {split.inputs.shape[0]}")
    assert pack_([a], 3).inputs.shape == (1, 3), (
        "an example whose 3 positions exactly fill a row of 3 must fit, not raise or split")
    try:
        pack_([a], 2)
    except ValueError:
        pass
    else:
        raise AssertionError("an example longer than a row must raise ValueError")
    # The decisive property: a packed example computes exactly what it computes alone.
    params = init_params(3, 0.5)
    logits, _ = forward(params, batch.inputs, batch.pos, batch.allowed)
    assert np.isfinite(logits).all(), "the forward pass produced nan: some position attends to nothing"
    alone = pack_([b], 8)
    alone_logits, _ = forward(params, alone.inputs, alone.pos, alone.allowed)
    drift = float(np.abs(logits[0, 3:7] - alone_logits[0, 0:4]).max())
    assert drift < 1e-9, (
        f"the second example's logits change by {drift:.2e} when it is packed behind another: "
        "the mask lets it see the first example, or its position ids did not restart")
    print("exercise 3 looks right: rows split exactly where they must, and a packed example's "
          "logits match running it alone to within 1e-9")

In [ ]:
if __name__ == "__main__":
    _try("exercise 3", _check_pack)

## 9. Break it: five planted bugs, seven fine-tunes

Your three pieces are now the **clean pipeline**. Each bug below swaps exactly one piece for a
broken version that plausible code could contain. None of them raises an error.

1. **Template drift.** Training renders the conversation without closing the user's turn with
   `END`; the server renders it with `END`, as the template says. Two code paths, one forgotten
   separator: the toy version of training with one chat template and serving with another.
2. **The prompt in the loss.** No assistant-only mask: every token is a target, so the model is
   also trained to write the user's side.
3. **The end token never learned.** The pad token is set equal to the end token (TRL's own
   documentation says it falls back to the EOS token when no pad token is set) and the loss
   ignores "padding" by comparing ids, so every real `END` is ignored too.
4. **The mask off by one.** The mask is built in the same coordinates as the *inputs* but the
   loss reads it in the coordinates of the *targets*, so every weight lands one token off.
5. **The packing leak.** Rows are packed, but with a plain causal mask over the whole row, so a
   conversation attends to whatever happens to sit in front of it.

Each bug is trained under identical conditions: the same pretrained base, the same training
conversations, the same sampling seed, the same number of steps, the same optimiser. Bugs 1 to 4
train with padded batches and are compared with a clean padded run; bug 5 needs packing, so it
is compared with a clean packed run. After each run the model answers every held-out prompt
by greedy decoding, using the *correct* serving template, and the cell counts what came back.
Read the table before you write a single test.

In [ ]:
SFT_STEPS = 800
SFT_BATCH = 32
SFT_SEED = 2
EVAL_EVERY = 100
MAX_NEW = 8
LOSS_LOOKS_FINE = 0.05     # nats: "a training loss that would not worry anyone"


def build_example_no_user_end(messages: list) -> tuple:
    """Bug 1: training drops the END that closes every non-assistant turn."""
    ids, supervised = build_example(messages)
    keep = [not (token == END and not learned) for token, learned in zip(ids, supervised)]
    return ([t for t, k in zip(ids, keep) if k], [s for s, k in zip(supervised, keep) if k])


def build_example_unmasked(messages: list) -> tuple:
    """Bug 2: no assistant-only mask, so every token is a target."""
    ids, _ = build_example(messages)
    return ids, [True] * len(ids)


def collate_pad_is_end(examples: list, max_len: int) -> Batch:
    """Bug 3: pad token == end token, and the loss 'ignores padding' by ignoring that id."""
    batch = collate(examples, max_len, pad_id=END)
    return batch._replace(weights=batch.weights * (batch.targets != END))


def collate_unshifted_mask(examples: list, max_len: int) -> Batch:
    """Bug 4: targets are shifted by one but the mask is not."""
    return collate(examples, max_len, shift_mask=False)


def pack_leaky(examples: list, max_len: int) -> Batch:
    """Bug 5: a plain causal mask over the whole row, so conversations see each other."""
    batch = pack(examples, max_len)
    n, t = batch.inputs.shape
    leaky = np.broadcast_to(np.tril(np.ones((t, t), dtype=bool)), (n, t, t)).copy()
    return batch._replace(allowed=leaky)


BUG_NAMES = ("template drift", "prompt in the loss", "end token unlearned", "mask off by one",
             "packing leak")


def make_zoo() -> dict:
    """The clean pipeline and one pipeline per planted bug, each a Pipeline."""
    clean = make_pipeline()
    return {"clean": clean,
            "template drift": clean._replace(build_example=build_example_no_user_end),
            "prompt in the loss": clean._replace(build_example=build_example_unmasked),
            "end token unlearned": clean._replace(collate=collate_pad_is_end),
            "mask off by one": clean._replace(collate=collate_unshifted_mask),
            "packing leak": clean._replace(pack=pack_leaky)}


def train_step(params: dict, state: dict, batch: Batch) -> float:
    """One optimiser step on a Batch. Returns the loss before the step."""
    logits, cache = forward(params, batch.inputs, batch.pos, batch.allowed)
    loss, _ = masked_loss(logits, batch.targets, batch.weights)
    adam_step(params, backward(params, cache, logits, batch.targets, batch.weights), state)
    return loss


def generate(params: dict, prompts: list, max_new: int = MAX_NEW) -> list:
    """Greedy decoding. Every prompt must have the same length. Each answer ends at its first
    END (included), or runs on to `max_new` tokens if the model never emits one."""
    ids = np.array(prompts, dtype=np.int64)
    n = len(ids)
    answers, finished = [[] for _ in range(n)], [False] * n
    for _ in range(max_new):
        t = ids.shape[1]
        allowed = np.broadcast_to(np.tril(np.ones((t, t), dtype=bool)), (n, t, t))
        logits, _ = forward(params, ids, np.tile(np.arange(t), (n, 1)), allowed)
        chosen = logits[:, -1].argmax(axis=-1)
        for j in range(n):
            if not finished[j]:
                answers[j].append(int(chosen[j]))
                finished[j] = int(chosen[j]) == END
        ids = np.concatenate([ids, chosen[:, None]], axis=1)
    return answers


def behaviour(params: dict, serve_render: Callable, payloads: list) -> dict:
    """Held-out behaviour: serve every payload with `serve_render`, decode greedily, count."""
    prompts = [serve_render([{"role": "user", "content": p}], True) for p in payloads]
    answers = generate(params, prompts)
    gold = [encode_text(p.upper()) for p in payloads]
    return {"n": len(payloads),
            "exact": sum(a == g + [END] for a, g in zip(answers, gold)),
            "content": sum(a[:len(g)] == g for a, g in zip(answers, gold)),
            "first": sum(a[0] == g[0] for a, g in zip(answers, gold)),
            "letters": sum(sum(len(a) > i and a[i] == g[i] for i in range(len(g)))
                           for a, g in zip(answers, gold)),
            "stopped": sum(END in a for a in answers),
            "mean_len": float(np.mean([len(a) for a in answers]))}


def finetune(base: dict, pipe: Pipeline, mode: str, steps: int = SFT_STEPS, seed: int = SFT_SEED):
    """Fine-tune a copy of `base` on the training conversations through `pipe`. `mode` is "pad"
    (pipe.collate) or "pack" (pipe.pack). Returns (params, losses, curve) where curve holds
    (step, held-out answers with the right letters) every EVAL_EVERY steps."""
    params = {k: v.copy() for k, v in base.items()}
    state, rng = adam_init(params), np.random.default_rng(seed)
    batcher = (lambda ex: pipe.collate(ex, PAD_LEN)) if mode == "pad" \
        else (lambda ex: pipe.pack(ex, PACK_LEN))
    losses, curve = [], []
    for step in range(1, steps + 1):
        picks = rng.integers(0, len(TRAIN_PAYLOADS), SFT_BATCH)
        examples = [pipe.build_example(conversation(TRAIN_PAYLOADS[i])) for i in picks]
        losses.append(train_step(params, state, batcher(examples)))
        if step % EVAL_EVERY == 0:
            curve.append((step, behaviour(params, pipe.serve_render, HELD_PAYLOADS)["content"]))
    return params, losses, curve


def neighbour_sensitivity(params: dict, pack_fn: Callable, payloads: list, seed: int = 9) -> tuple:
    """How much does what sits IN FRONT of a prompt change the model's first answer token?

    Each prompt is packed alone, then packed behind a random training conversation, using
    `pack_fn`. Returns the largest and the mean total-variation distance between the two
    next-token distributions: 0 means the neighbour is invisible."""
    rng = np.random.default_rng(seed)
    distances = []
    for payload in payloads:
        prompt = render_chat([{"role": "user", "content": payload}], True) + [PAD]
        neighbour = render_chat(conversation(TRAIN_PAYLOADS[int(rng.integers(len(TRAIN_PAYLOADS)))]))
        answers = []
        for sequences in ([prompt], [neighbour, prompt]):
            batch = pack_fn([(s, [False] * len(s)) for s in sequences], PACK_LEN)
            logits, _ = forward(params, batch.inputs, batch.pos, batch.allowed)
            answers.append(softmax(logits[0, batch.spans[0][-1][1] - 1]))
        distances.append(0.5 * float(np.abs(answers[0] - answers[1]).sum()))
    return max(distances), float(np.mean(distances))


def prompt_loss_share(pipe: Pipeline) -> float:
    """Fraction of the loss weight that lands on prompt tokens: everything before the first
    reply token of a plain two-message conversation."""
    chat = conversation(TRAIN_PAYLOADS[0])
    batch = pipe.collate([pipe.build_example(chat)], PAD_LEN)
    prompt_targets = len(render_chat(chat[:1], True)) - 1
    return float(batch.weights[0, :prompt_targets].sum() / max(float(batch.weights.sum()), 1.0))


def steps_to(curve: list, fraction: float, n: int):
    """The first checkpoint step at which at least `fraction` of the held-out answers had the
    right letters, or None if it never did."""
    return next((step for step, right in curve if right >= fraction * n), None)


def at_step(step) -> str:
    """'at step 600', or 'not within N steps' when no checkpoint got there."""
    return f"at step {step}" if step is not None else f"not within {SFT_STEPS} steps"

In [ ]:
def run_zoo() -> dict:
    """Fine-tune the five buggy pipelines and the two clean ones they are compared with, measure
    each, print the table."""
    zoo = make_zoo()
    plan = [("clean, padded", "clean", "pad"), ("template drift", "template drift", "pad"),
            ("prompt in the loss", "prompt in the loss", "pad"),
            ("end token unlearned", "end token unlearned", "pad"),
            ("mask off by one", "mask off by one", "pad"),
            ("clean, packed", "clean", "pack"), ("packing leak", "packing leak", "pack")]
    results = {"base": {"behaviour": behaviour(BASE, zoo["clean"].serve_render, HELD_PAYLOADS),
                        "loss": float("nan"), "curve": [], "sensitivity": None, "share": None}}
    for name, key, mode in plan:
        pipe = zoo[key]
        params, losses, curve = finetune(BASE, pipe, mode)
        results[name] = {
            "behaviour": behaviour(params, pipe.serve_render, HELD_PAYLOADS),
            "loss": float(np.mean(losses[-50:])), "curve": curve,
            "sensitivity": (neighbour_sensitivity(params, pipe.pack, HELD_PAYLOADS[:64])
                            if mode == "pack" else None),
            "share": prompt_loss_share(pipe) if mode == "pad" else None}
    n = results["base"]["behaviour"]["n"]
    print(f"{SFT_STEPS} steps of {SFT_BATCH} conversations each, from the same base, scored on "
          f"{n} held-out prompts (counts are out of {n}; letters out of {PAYLOAD_LEN * n}):\n")
    print(f"{'run':21s} {'final loss':>10s} {'exact':>6s} {'letters':>8s} {'first':>6s} "
          f"{'stopped':>8s} {'mean len':>9s}")
    for name, r in results.items():
        b = r["behaviour"]
        loss = "n/a" if r["loss"] != r["loss"] else f"{r['loss']:.4f}"
        print(f"{name:21s} {loss:>10s} {b['exact']:6d} {b['letters']:8d} {b['first']:6d} "
              f"{b['stopped']:8d} {b['mean_len']:9.2f}")
    return results


def read_zoo(results: dict) -> None:
    """Say what the table shows, with every number and every comparison word computed."""
    clean, packed = results["clean, padded"], results["clean, packed"]
    n = clean["behaviour"]["n"]
    print("\nwhat each bug did, against its clean run (computed from the table):")
    r = results["template drift"]
    print(f"  template drift:      final loss {r['loss']:.4f} against {clean['loss']:.4f} clean, "
          f"yet {r['behaviour']['exact']} of {n} served answers were exact "
          f"(clean: {clean['behaviour']['exact']})")
    r = results["prompt in the loss"]
    hit = lambda run: steps_to(run["curve"], 0.9, n)
    print(f"  prompt in the loss:  {r['behaviour']['exact']} of {n} exact "
          f"(clean: {clean['behaviour']['exact']}); {r['share']:.0%} of its loss weight sat on "
          f"prompt tokens (clean: {clean['share']:.0%}); 90% of answers had the right letters: "
          f"{at_step(hit(r))} (clean: {at_step(hit(clean))})")
    r = results["end token unlearned"]
    print(f"  end token unlearned: stopped {r['behaviour']['stopped']} of {n} "
          f"(clean: {clean['behaviour']['stopped']}), mean generation {r['behaviour']['mean_len']:.2f} "
          f"tokens against {clean['behaviour']['mean_len']:.2f}, with the right letters in "
          f"{r['behaviour']['content']} of {n}")
    r = results["mask off by one"]
    print(f"  mask off by one:     first reply token right in {r['behaviour']['first']} of {n} "
          f"(clean: {clean['behaviour']['first']}), letters right "
          f"{r['behaviour']['letters']} of {PAYLOAD_LEN * n} (clean: {clean['behaviour']['letters']})")
    r = results["packing leak"]
    print(f"  packing leak:        {r['behaviour']['exact']} of {n} exact alone "
          f"(clean packed: {packed['behaviour']['exact']}); with a different conversation packed in "
          f"front, the first answer token's distribution moved by up to {r['sensitivity'][0]:.3f} "
          f"(mean {r['sensitivity'][1]:.3f}) against {packed['sensitivity'][0]:.3f} "
          f"(mean {packed['sensitivity'][1]:.3f}) for the clean packer; its final training loss was "
          f"{r['loss']:.4f} against {packed['loss']:.4f} for the clean packed run")
    hidden = [name for name in BUG_NAMES[:4]
              if results[name]["loss"] <= clean["loss"] + LOSS_LOOKS_FINE
              and results[name]["behaviour"]["exact"] < clean["behaviour"]["exact"]]
    print(f"\nbugs whose final training loss is within {LOSS_LOOKS_FINE} nats of the clean run's, "
          f"or below it, while exact match is lower: {hidden if hidden else 'none'}")
    if hidden:
        print("so the loss curve cannot be the test for those: it scores whatever the pipeline "
              "chose to learn, and a pipeline that learns the wrong thing reports a fine loss "
              "for it")

In [ ]:
RESULTS = None
if __name__ == "__main__":
    def _run_and_read() -> None:
        global RESULTS
        RESULTS = run_zoo()
        read_zoo(RESULTS)

    _try("the seven fine-tunes", _run_and_read, needs=_FOR_PIPELINE)

## 10. From symptoms to tests

The table is the symptom. A test is what you run *before training*, on the pipeline's own
output, so that the bug stops the run instead of costing it. Every test below has the same
shape: it takes a whole `Pipeline` and returns **True when it finds the bug**, False when the
pipeline is sound. A test that cries wolf blocks every run; a test that stays silent is
worse than none, because it is trusted. So each check you run tests both directions.

Two rules for writing them:

- **Probe with conversations of your own** (letters `a` to `h` are safe content), and probe more
  than the simplest case: real pipelines go wrong in a second turn or after a system message.
  Find what you are looking for in the ids *the pipeline produced*, not in ids you computed
  yourself: a pipeline with one bug must not make a test for another bug misread its positions.
- **A test sees a pipeline.** Four of the five never run the model at all, the fifth only runs
  it forward, and none may train anything.

The cell below gives you more bug shapes than the five in the table. The checks run every
detector against all of them, because a bug is a family and a test that catches one member of
the family is only a start. It also builds a tiny helper for exercise 8, so run it first.

In [ ]:
def serve_without_generation_prompt(messages: list, add_generation_prompt: bool = True) -> list:
    """Bug 1, variant: the server ignores add_generation_prompt."""
    return render_chat(messages, False)


def build_example_forgets_history_end(messages: list) -> tuple:
    """Bug 1, variant: training drops the END after every assistant turn except the last."""
    ids, supervised = build_example(messages)
    ends = [i for i, (t, s) in enumerate(zip(ids, supervised)) if t == END and s]
    drop = set(ends[:-1])
    return ([t for i, t in enumerate(ids) if i not in drop],
            [s for i, s in enumerate(supervised) if i not in drop])


def build_example_masks_only_first_message(messages: list) -> tuple:
    """Bug 2, variant: only the first message is masked; every later token is a target."""
    ids, _ = build_example(messages)
    first_end = ids.index(END)
    return ids, [i > first_end for i in range(len(ids))]


def build_example_never_supervises_end(messages: list) -> tuple:
    """Bug 3, variant: the loss mask leaves out every END, assistant turns included."""
    ids, supervised = build_example(messages)
    return ids, [s and t != END for t, s in zip(ids, supervised)]


def build_example_last_end_only(messages: list) -> tuple:
    """Bug 3, variant: only the final assistant turn's END is learned."""
    ids, supervised = build_example(messages)
    ends = [i for i, (t, s) in enumerate(zip(ids, supervised)) if t == END and s]
    drop = set(ends[:-1])
    return ids, [s and i not in drop for i, s in enumerate(supervised)]


def build_example_skips_first_reply_token(messages: list) -> tuple:
    """Bug 4, variant: every reply is supervised starting one token late."""
    ids, supervised = build_example(messages)
    starts = {i for i in range(len(ids)) if supervised[i] and (i == 0 or not supervised[i - 1])}
    return ids, [s and i not in starts for i, s in enumerate(supervised)]


def build_example_omits_marker(messages: list) -> tuple:
    """Bug 1, variant: training leaves out the ASSISTANT marker that starts the last reply."""
    ids, supervised = build_example(messages)
    drop = {i for i, t in enumerate(ids) if t == ASSISTANT}
    drop = set(sorted(drop)[-1:])
    return ([t for i, t in enumerate(ids) if i not in drop],
            [s for i, s in enumerate(supervised) if i not in drop])


def build_example_learns_marker(messages: list) -> tuple:
    """Bug 2, variant: the mask also covers the ASSISTANT marker, one prompt token."""
    ids, supervised = build_example(messages)
    return ids, [s or t == ASSISTANT for t, s in zip(ids, supervised)]


def build_example_learns_later_user_text(messages: list) -> tuple:
    """Bug 2, variant: the text of every user turn after the first reply is learned."""
    ids, supervised = build_example(messages)
    first_reply = ids.index(ASSISTANT) if ASSISTANT in ids else len(ids)
    return ids, [s or (i > first_reply and t < len(TEXT_CHARS))
                 for i, (t, s) in enumerate(zip(ids, supervised))]


def build_example_final_reply_unended(messages: list) -> tuple:
    """Bug 3, variant: the template forgets the END after the last assistant message."""
    ids, supervised = build_example(messages)
    ends = [i for i, (t, s) in enumerate(zip(ids, supervised)) if t == END and s]
    drop = set(ends[-1:])
    return ([t for i, t in enumerate(ids) if i not in drop],
            [s for i, s in enumerate(supervised) if i not in drop])


def build_example_later_replies_skip_first_token(messages: list) -> tuple:
    """Bug 4, variant: only the second and later replies are supervised one token late."""
    ids, supervised = build_example(messages)
    starts = [i for i in range(len(ids)) if supervised[i] and (i == 0 or not supervised[i - 1])]
    late = set(starts[1:])
    return ids, [s and i not in late for i, s in enumerate(supervised)]


def pack_boundary_leak(examples: list, max_len: int) -> Batch:
    """Bug 5, variant: the first token of each example may also see the last token before it."""
    batch = pack(examples, max_len)
    allowed = batch.allowed.copy()
    for r, row_spans in enumerate(batch.spans):
        for start, _ in row_spans[1:]:
            allowed[r, start, start - 1] = True
    return batch._replace(allowed=allowed)


def pack_positions_continue(examples: list, max_len: int) -> Batch:
    """Bug 5, variant: the mask is right but position ids keep counting across examples."""
    batch = pack(examples, max_len)
    pos = np.zeros_like(batch.pos)
    for r, row_spans in enumerate(batch.spans):
        for start, end in row_spans:
            pos[r, start:end] = np.arange(start, end)
    return batch._replace(pos=pos)


def pack_attends_to_row_start(examples: list, max_len: int) -> Batch:
    """Bug 5, variant: the mask is block-diagonal except that everything may also see position 0."""
    batch = pack(examples, max_len)
    allowed = batch.allowed.copy()
    allowed[:, :, 0] = True
    return batch._replace(allowed=allowed)


def make_variants() -> dict:
    """For each bug, every pipeline shape of it that this notebook knows: (label, Pipeline)."""
    zoo = make_zoo()
    clean = zoo["clean"]
    return {
        "template drift": [
            ("training drops the user END", zoo["template drift"]),
            ("server forgets the generation prompt",
             clean._replace(serve_render=serve_without_generation_prompt)),
            ("earlier assistant turns lose their END in training",
             clean._replace(build_example=build_example_forgets_history_end)),
            ("training leaves out the last reply's ASSISTANT marker",
             clean._replace(build_example=build_example_omits_marker))],
        "prompt in the loss": [
            ("no mask at all", zoo["prompt in the loss"]),
            ("only the first message masked",
             clean._replace(build_example=build_example_masks_only_first_message)),
            ("the ASSISTANT marker is learned",
             clean._replace(build_example=build_example_learns_marker)),
            ("user text after the first reply is learned",
             clean._replace(build_example=build_example_learns_later_user_text))],
        "end token unlearned": [
            ("pad id equals end id", zoo["end token unlearned"]),
            ("the mask leaves out every END",
             clean._replace(build_example=build_example_never_supervises_end)),
            ("only the last assistant END is learned",
             clean._replace(build_example=build_example_last_end_only)),
            ("the last reply is never ended",
             clean._replace(build_example=build_example_final_reply_unended))],
        "mask off by one": [
            ("targets shifted, mask not", zoo["mask off by one"]),
            ("replies supervised one token late",
             clean._replace(build_example=build_example_skips_first_reply_token)),
            ("only later replies skip their first token",
             clean._replace(build_example=build_example_later_replies_skip_first_token))],
        "packing leak": [
            ("plain causal mask over the row", zoo["packing leak"]),
            ("position ids keep counting", clean._replace(pack=pack_positions_continue)),
            ("everything also sees the row's first token",
             clean._replace(pack=pack_attends_to_row_start)),
            ("a one-token leak across the example boundary",
             clean._replace(pack=pack_boundary_leak))]}


def sample_logits(pipe: Pipeline, params: dict, token_lists: list) -> list:
    """Pack the token sequences into ONE row with `pipe.pack`, run the model once, and return
    each sequence's logits: a list of (len(ids) - 1, VOCAB_SIZE) arrays, in the order given.

    This is the helper for exercise 8. It builds the packed row through the pipeline under test,
    so a packer with a leaking mask or wrong position ids shows up in what it returns."""
    batch = pipe.pack([(ids, [False] * len(ids)) for ids in token_lists], PACK_LEN)
    assert len(batch.spans) == 1, "the sequences do not fit one row of PACK_LEN positions"
    logits, _ = forward(params, batch.inputs, batch.pos, batch.allowed)
    return [logits[0, start:end] for start, end in batch.spans[0]]


def _verdicts(detector: Callable, bug: str, note: str = "") -> None:
    """The shared body of every detector check: silent on the clean pipeline, True on every shape
    of its own bug."""
    zoo, variants = make_zoo(), make_variants()

    def verdict(pipe: Pipeline) -> bool:
        answer = detector(pipe)
        assert isinstance(answer, (bool, np.bool_)), (
            f"a detector must return True or False, it returned {answer!r}")
        return bool(answer)

    assert not verdict(zoo["clean"]), (
        "the detector returned True for the CLEAN pipeline: a false alarm blocks every training "
        "run, so a test that cannot stay quiet on a correct pipeline is not a test." + note)
    missed = [label for label, pipe in variants[bug] if not verdict(pipe)]
    assert not missed, (f"the detector did not fire on: {missed}. Each is the same bug in a "
                        "different shape, and a test has to catch the family, not one member." + note)

## 11. Exercise 4 · `detects_template_drift(pipe)`

The template drift test is the one the whole industry needs most, because the two code paths
really are different code: training renders through one function, serving through another (a
library, a server's template file, a hand-written f-string).

The property is a **prefix property**. `pipe.build_example(messages)` gives the ids training
saw for a whole conversation. `pipe.serve_render(messages[:k], True)` gives the ids the server
sends when it is about to ask for the assistant message at index `k`. The served prompt must be
*exactly the beginning of the training sequence*, and it must end on the `ASSISTANT` marker.
Check it for **every** assistant message, because a later reply's prompt contains the earlier
replies, and an earlier reply may have been rendered differently in training than in the history.

<details><summary>💡 Hint 1 — what to think about</summary>

For each assistant message, what is the prompt the server would build, and what is the training
sequence that reply came from? Say, in one sentence, how the first should relate to the second.
That sentence has two parts: one about the *beginning* of the ids and one about the *end* of
the prompt. A single probe conversation proves little; what second kind of conversation can
expose a difference that a single reply hides?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Write at least two probe conversations, one with a single exchange and one that opens with a
system message and has two exchanges. For each assistant message, render the prompt with the
server's function and compare it, token for token, with the same-length beginning of the
training ids, and also require its final token to be the assistant marker. Return True on the
first disagreement and False only after every probe agrees.

</details>

In [ ]:
def detects_template_drift(pipe: Pipeline) -> bool:
    """True if the prompt the server builds is not the beginning of what training saw.

    Uses pipe.build_example (training) and pipe.serve_render (serving). For every assistant
    message in a probe conversation, the served prompt for it must equal the first len(prompt)
    ids of the training sequence AND end with the ASSISTANT marker. Probe single- and
    multi-turn conversations. Do not train anything.

    Example:
        >>> detects_template_drift(make_pipeline())
        False

    Returns:
        True when the pipeline has the bug, False when every probe agrees.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_template_drift() -> None:
    _verdicts(detects_template_drift, "template drift")
    print("exercise 4 looks right: silent on the clean pipeline, and it fires on all "
          f"{len(make_variants()['template drift'])} shapes of template drift")

In [ ]:
if __name__ == "__main__":
    _try("exercise 4", _check_template_drift, needs=_FOR_PIPELINE)

## 12. Exercise 5 · `detects_prompt_in_loss(pipe)`

The loss mask is only visible after batching: `build_example` says which ids are learned, but
the *weights* the model actually trains with are what `collate` produces, after shifting and
padding. Test the weights, not the mask. For a probe conversation, build the example, collate
it (use `max_len=PACK_LEN` so a multi-turn probe fits), and ask: is any prompt token a target
with weight above zero? A **reply** is what follows an `ASSISTANT` marker, up to and including
the `END` that closes it. Every other id, the marker itself included, is a **prompt token**.

Remember what a weight scores: `weights[0, t]` is the weight on the *target* `targets[0, t]`,
which is token `t + 1` of the example.

<details><summary>💡 Hint 1 — what to think about</summary>

Which index of the weights row belongs to the token at example index `i`? Work out the offset
on paper with a tiny example before you code it. Then decide which example indices are prompt
tokens, reading the ids the pipeline gave you, and ask whether each is weighted. Index 0 can
never be a target, so it needs no check.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Walk the ids of the example the pipeline built, keeping one flag that says whether you are
currently inside a reply: it turns on at an `ASSISTANT` marker (which is itself prompt) and off
after the `END` that closes the reply. Every id seen while the flag is off is prompt. For each
prompt id that is not the very first, look at the weight stored one position earlier. Probe a
conversation with more than one exchange as well.

</details>

In [ ]:
def detects_prompt_in_loss(pipe: Pipeline) -> bool:
    """True if any prompt token is a target with loss weight above zero.

    Build each probe with pipe.build_example, batch it with pipe.collate(examples, PACK_LEN), and
    read batch.weights. A reply is the ids after an ASSISTANT marker up to and including the END
    that closes it; every other id, the ASSISTANT marker included, is prompt. Find them in the ids
    the pipeline itself produced. weights[0, t] scores token t + 1. Probe several conversations.

    Example:
        >>> detects_prompt_in_loss(make_pipeline())
        False

    Returns:
        True when the bug is present, False when no prompt token carries weight.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_prompt_in_loss() -> None:
    _verdicts(detects_prompt_in_loss, "prompt in the loss")
    print("exercise 5 looks right: no weight on a prompt token passes, and any prompt token "
          "with weight is caught")

In [ ]:
if __name__ == "__main__":
    _try("exercise 5", _check_prompt_in_loss, needs=_FOR_PIPELINE)

## 13. Exercise 6 · `detects_unlearned_stop(pipe)`

A model stops generating because it learned to emit `END` at the right moment, and it learned
that only if `END` was a *target with weight* in training. Lose that and the model answers
correctly and then keeps going, which in a chat product shows up as replies that never end, or
that wander into an invented next user turn.

The `END` token can be lost in three places: the template never appends it, the mask leaves it
out, or the collator discards it. Test the final weights, as in exercise 5: for every assistant
message in a probe, the `END` that closes it must be a target with weight above zero.

<details><summary>💡 Hint 1 — what to think about</summary>

Where in the ids does the `END` that closes a reply sit, and which weight scores it? The same
offset arithmetic as exercise 5 applies, but the weight must be present rather than absent.
Counting how many `END` targets carry weight is tempting: ask whether a count can be fooled
when some other `END` is learned by mistake. And what if a reply has no `END` at all?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Probe a conversation with several assistant turns, because some bugs lose all of the end tokens
but the last. In the ids the pipeline built, find each `ASSISTANT` marker and the first `END`
after it. If there is none, return True. Otherwise the weight that scores that `END` is stored
one position before the `END`'s own index; if it is zero for any reply, return True.

</details>

In [ ]:
def detects_unlearned_stop(pipe: Pipeline) -> bool:
    """True if the END that closes any assistant message is not a target with weight above zero.

    Build each probe with pipe.build_example, batch it with pipe.collate(examples, PACK_LEN), and
    read batch.weights. In the ids the pipeline produced, the END that closes a reply is the first
    END after its ASSISTANT marker; a reply with no END at all is the bug too. weights[0, t]
    scores token t + 1. Probe a conversation with several assistant turns.

    Example:
        >>> detects_unlearned_stop(make_pipeline())
        False

    Returns:
        True when some assistant END is not learned, False when every one is.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_unlearned_stop() -> None:
    _verdicts(detects_unlearned_stop, "end token unlearned")
    print("exercise 6 looks right: every assistant END must carry weight, and a missing one is "
          "caught whichever stage lost it")

In [ ]:
if __name__ == "__main__":
    _try("exercise 6", _check_unlearned_stop, needs=_FOR_PIPELINE)

## 14. Exercise 7 · `detects_shifted_mask(pipe)`

An off-by-one in the mask is the most deceptive of the five: the mask *looks* right when you
print it over the ids, and the model trains without complaint. The consequence is local and
exact. If the mask is a position too early the first reply token is not learned, and the
model's first token is a guess. If it is a position too late, a token that should be a prompt
is learned instead.

Test the property that fails first: **the first token of every reply must be a target with
weight above zero**. A reply's first token is the id right after its `ASSISTANT` marker; the
weight that scores it is the one stored at the *marker's* index. (A reply with no text has
only its `END` after the marker; skip it, exercise 6 covers that.)

<details><summary>💡 Hint 1 — what to think about</summary>

Index the first reply token in the ids, then index the weight that scores it, and notice they
differ by exactly the shift between inputs and targets. Draw both rows for a short conversation
before you write the loop. Two different pipelines can fail this test: one whose collator shifts
the targets but not the weights, and one whose mask builder starts a token late. Does your test
see both?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Probe a multi-turn conversation, because the second reply is a second chance to be off. Walk the
ids the pipeline built and stop at each `ASSISTANT` marker whose next id is text rather than
`END`. The weight stored at the marker's own index must be above zero. Return True the moment
one is not.

</details>

In [ ]:
def detects_shifted_mask(pipe: Pipeline) -> bool:
    """True if the first token of any reply is not a target with weight above zero.

    Build each probe with pipe.build_example, batch it with pipe.collate(examples, PACK_LEN), and
    read batch.weights. weights[0, t] scores token t + 1, so the first reply token (the id after
    an ASSISTANT marker at index m) is scored by weights[0, m]. Skip replies with no text, whose
    first id is already their END. Probe several conversations.

    Example:
        >>> detects_shifted_mask(make_pipeline())
        False

    Returns:
        True when some reply's first token is not learned, False when all are.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_shifted_mask() -> None:
    _verdicts(detects_shifted_mask, "mask off by one")
    print("exercise 7 looks right: the first reply token carries weight in a sound pipeline, "
          "and a reply whose first token lost its weight is caught, whether the collator or the "
          "mask builder dropped it")

In [ ]:
if __name__ == "__main__":
    _try("exercise 7", _check_shifted_mask, needs=_FOR_PIPELINE)

## 15. Exercise 8 · `detects_cross_sample_leak(pipe)`

The packing test is the only one that has to run the model, and it is the best test in the
lesson because it states the property directly: **a conversation's logits must not depend on
what else is in the row**. Pack a conversation `B` behind a conversation `A`; the logits for
`B`'s tokens must equal what the model computes for `B` packed alone. If a different `A2` sits
in front instead, they must still be the same.

`sample_logits(pipe, params, [A, B])` (given, in section 10) packs through `pipe.pack`, runs the
model once and hands back each conversation's logits. Use a **random** model, not a trained one
and not a zero-filled one, and of moderate size: you want every weight to matter, but with very
large weights attention saturates into one-hot rows and a small leak can vanish. Compare with a
small tolerance, never `==`, because the same arithmetic done on arrays of different shapes can
differ in the last bit.

<details><summary>💡 Hint 1 — what to think about</summary>

There are two different ways for a packed row to corrupt `B`: it can see tokens it should not,
and it can be told the wrong positions. Which comparison exposes each? One comparison (what
`B` computes behind `A` against what it computes behind a different `A2`) is blind to some
bugs, because both sides could be equally wrong. What is the one reference that cannot be
wrong in that way?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Make a random-weight model with `init_params`, with a scale you have thought about: neither
so small that nothing matters nor so large that attention goes one-hot. Build three token
sequences with `render_chat`: two different neighbours and one conversation `B`. Get `B`'s
logits alone, behind the first neighbour and behind the second. Take the largest absolute
difference between the alone logits and each of the others, and return True when it exceeds a
tolerance far above rounding noise and far below any real leak.

</details>

In [ ]:
def detects_cross_sample_leak(pipe: Pipeline) -> bool:
    """True if a conversation's logits change with what is packed next to it.

    Use sample_logits(pipe, params, [...]) with a random model (init_params with a scale of 0.5
    or so). Compare the logits of one conversation packed alone with the same conversation packed
    behind two different neighbours. Use a tolerance, not exact equality.

    Example:
        >>> detects_cross_sample_leak(make_pipeline())
        False

    Returns:
        True when the packed logits differ beyond rounding, False when they agree.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_cross_sample_leak() -> None:
    _verdicts(detects_cross_sample_leak, "packing leak")
    # Again with a model whose weights are small: a leak moves its logits much less, so a
    # tolerance set just below the size of a big leak stops seeing it.
    original = globals()["init_params"]
    globals()["init_params"] = lambda seed=0, scale=INIT_SCALE: original(seed, min(scale, 0.05))
    try:
        _verdicts(detects_cross_sample_leak, "packing leak",
                  note=" (This run asked for a model with small weights, where a leak moves the "
                       "logits only a little: put the tolerance just above rounding noise.)")
    finally:
        globals()["init_params"] = original
    print("exercise 8 looks right: a packed conversation computes what it computes alone, and "
          "a leaking mask, restarted positions that do not restart, and a stray look at the row "
          "start are all caught")

In [ ]:
if __name__ == "__main__":
    _try("exercise 8", _check_cross_sample_leak, needs=_FOR_PIPELINE)

## 16. The preflight: five tests, run before any training

Your five tests are now a **preflight**: a function that takes a pipeline, runs every test, and
lists the bugs it found. `train_if_clean` refuses to start a fine-tune when the list is not
empty. The next cell runs it on the clean pipeline and on all five buggy ones and prints which
tests fired on which pipeline. Every mark in that matrix is computed from your five functions.

In [ ]:
PREFLIGHT = (("template drift", "detects_template_drift"),
             ("prompt in the loss", "detects_prompt_in_loss"),
             ("end token unlearned", "detects_unlearned_stop"),
             ("mask off by one", "detects_shifted_mask"),
             ("packing leak", "detects_cross_sample_leak"))


class PipelineBug(Exception):
    """Raised by `train_if_clean` when a preflight test finds a bug."""


def preflight(pipe: Pipeline) -> list:
    """Run every test on `pipe`. Returns the names of the bugs found: an empty list means clean."""
    return [bug for bug, function in PREFLIGHT if globals()[function](pipe)]


def train_if_clean(base: dict, pipe: Pipeline, mode: str):
    """Refuse to train a pipeline the preflight objects to."""
    found = preflight(pipe)
    if found:
        raise PipelineBug("preflight found: " + ", ".join(found))
    return finetune(base, pipe, mode)


def show_preflight() -> None:
    zoo = make_zoo()
    fired = {name: preflight(pipe) for name, pipe in zoo.items()}
    short = {"template drift": "drift", "prompt in the loss": "prompt", "end token unlearned": "stop",
             "mask off by one": "shift", "packing leak": "leak"}
    print(f"{'pipeline':22s}" + "".join(f"{short[b]:>8s}" for b in BUG_NAMES))
    for name, found in fired.items():
        print(f"{name:22s}" + "".join(f"{'FIRES' if b in found else '.':>8s}" for b in BUG_NAMES))
    caught = [b for b in BUG_NAMES if fired[b]]
    own = [b for b in BUG_NAMES if b in fired[b]]
    extra = {b: [x for x in fired[b] if x != b] for b in BUG_NAMES if [x for x in fired[b] if x != b]}
    print(f"\nthe clean pipeline raised {len(fired['clean'])} alarms; "
          f"{len(caught)} of {len(BUG_NAMES)} bugs raised at least one; "
          f"{len(own)} of {len(BUG_NAMES)} were caught by their own test")
    print("alarms from a test other than the bug's own: "
          + ("; ".join(f"{b} -> {', '.join(x)}" for b, x in extra.items()) if extra else "none"))
    if extra:
        print("(an alarm outside the diagonal is legitimate when that pipeline also breaks the "
              "property the other test checks, and a false alarm when the test is wrong: look "
              "at which)")
    for bug in BUG_NAMES:
        try:
            train_if_clean(BASE, zoo[bug], "pack" if bug == "packing leak" else "pad")
            print(f"  {bug}: the fine-tune STARTED")
        except PipelineBug as refusal:
            print(f"  {bug}: refused before step 1 ({refusal})")
    _, _, curve = train_if_clean(BASE, zoo["clean"], "pad")
    print(f"  clean: trained, with {curve[-1][1]} of {len(HELD_PAYLOADS)} held-out answers right "
          "at the last checkpoint")


if __name__ == "__main__":
    _try("the preflight matrix", show_preflight,
         needs=tuple(label for label, _, _ in _EXERCISES))

## 17. Self-check

1. The model is fine-tuned on `<|user|> ... <|end|> <|assistant|> ... <|end|>`, but the serving
   code sends `<|user|> ... <|end|>` and stops, forgetting the generation prompt. What is the
   most accurate description of what the model now faces?
   - (a) Nothing unusual: the assistant marker is decoration the model can infer.
   - (b) A prompt that ends at a place where training never asked for a reply to begin, so the
         model's next-token behaviour there was never trained as the start of an answer.
   - (c) A prompt that is too short for attention to work.
   - (d) The same task, only slower.

2. A team sets the pad token equal to the end-of-sequence token and builds its loss mask by
   ignoring every position whose id equals the pad id. Training loss falls smoothly. What is
   the most likely behaviour after fine-tuning?
   - (a) Answers that are cut short after one token.
   - (b) Gibberish, because the embeddings collide.
   - (c) Right-looking answers that never stop on their own, running on until a length limit.
   - (d) The model refuses every request.

3. A packed row holds three conversations, with a plain causal mask over the whole row. Which
   statement is true?
   - (a) The second conversation's outputs depend on the first one's tokens, so the same prompt can
         get a different answer depending on what it was packed behind.
   - (b) Only the loss is affected; the outputs are the same as if each were run alone.
   - (c) Only the first conversation is affected.
   - (d) Padding between conversations makes the mask irrelevant.

4. The mask marks which token ids are replies. The loss reads targets, which are the ids shifted
   left by one. If the mask is not shifted too, what goes wrong?
   - (a) Nothing: mask and targets line up by construction.
   - (b) Training becomes faster but less stable.
   - (c) The gradient becomes `nan`.
   - (d) The loss lands on the wrong tokens: the first reply token is skipped and a token past
         the end of the reply is learned instead.

5. A fine-tune with a template mismatch reports a training loss as low as a clean one. Why is
   that unsurprising?
   - (a) Low loss always means the model is correct.
   - (b) The loss measures prediction of the sequences it was trained on, rendered one way. It
         says nothing about prompts rendered another way at serving time.
   - (c) The optimiser compensates for template differences.
   - (d) The held-out set leaked into training.

Answers, with reasoning, come with this lesson's worked solution when you enrol on Synapsa.

In [ ]:
# Question 4, measured rather than taken on trust: which targets does the loss actually score, for
# the same hand-written example, when the mask is shifted with the targets and when it is not?
def _demo_mask_coordinates() -> None:
    ids = [USER] + encode_text("ab") + [END, ASSISTANT] + encode_text("AB") + [END]
    learned = [False] * 5 + [True] * 3
    reply, exact = ids[5:], []
    for name, batcher in (("shifted like the targets", collate),
                          ("not shifted", collate_unshifted_mask)):
        batch = batcher([(ids, learned)], len(ids))   # one padding position after the reply's END
        scored = [int(token) for token, weight in zip(batch.targets[0], batch.weights[0])
                  if weight > 0]
        first = reply[0] in scored
        if scored == reply:
            exact.append(name)
        print(f"  mask {name:25s} scores {pretty(scored):16s} first reply letter scored: {first}")
    print(f"the reply is {pretty(reply)}; the mask that scores exactly that: "
          f"{', '.join(exact) if exact else 'neither'}")


if __name__ == "__main__":
    _demo_mask_coordinates()

## 18. Common mistakes

- **Two renderers for one template.** Training formats through a library; serving formats with a
  hand-written string. Test the prefix property (exercise 4) with the real functions, not copies.
- **Forgetting the generation prompt at serving.** Without the assistant marker the model has
  to guess that it is its turn. Hugging Face's `apply_chat_template` has the argument for exactly
  this; the failure is not passing it.
- **`pad_token = eos_token`, then masking by token id.** Reasonable alone, fatal together: every
  real end-of-turn token is treated as padding and dropped from the loss. Mask padding by
  *position* (the example's length), or keep a pad id distinct from the end id.
- **A mask in the wrong coordinates.** The mask describes input ids; the loss reads targets,
  which are the inputs shifted by one. Shift the mask exactly like the targets.
- **Masking by searching for the word "assistant".** Templates change with every model release:
  Qwen3.5 inserts an empty thinking block after the header, SmolLM3 brackets replies with
  generation keywords. Build the mask from the structure the template renders, and test it.
- **Supervising the stop but not the start, or the start but not the stop.** The `END` and the
  first reply token are the two boundaries of the reply, and each is where an off-by-one lands.
- **Packing with only a causal mask.** The mask must be block-diagonal and position ids must
  restart. As of 22 October 2026 TRL's documentation contrasts packing, which can combine parts
  of different samples in one row, with padding-free batching, which it says keeps every
  sequence complete and intact; this bug is the same cross-example leakage at toy scale.
- **Probing with one-turn conversations only.** Several of these bugs exist only from the second
  turn on, or after a system message.
- **Trusting the training loss.** It scores whatever the pipeline chose to supervise, so a
  pipeline that supervises the wrong tokens reports a healthy loss on them.

## 19. What this lesson does NOT cover

- **Real templates are bigger.** Tool calls, reasoning traces, images and multiple system
  messages each add markers and rules. The five properties tested here still hold; the probes
  that exercise them get longer.
- **Real tokenisers.** Here a conversation is rendered straight to ids. With a real tokeniser,
  formatting a conversation to a string and then tokenising that string can duplicate special
  tokens such as the start-of-sequence token. Hugging Face's documentation warns about this
  explicitly, and it is a sixth silent bug that none of the five tests here would see.
- **Truncation.** A conversation longer than the context is cut somewhere, and where it is cut
  decides whether the reply's `END` survives.
- **Other packing strategies.** As of 2 October 2026 TRL documents three (`bfd`, `bfd_split`,
  `wrapped`); it says `wrapped` cuts one stream into fixed-length blocks and "may mix unrelated
  examples", which gives up the invariance you tested, by design.
- **Bigger models.** This lesson does not measure what these bugs do to a large model; a bug
  whose symptoms are harder to see is no less present, which is why the tests run first.

In [ ]:
# The closing recap. Every figure is recomputed here, not quoted from the prose above.
if __name__ == "__main__":
    print("--- what you built, measured ---")
    print(f"  parameters in the model  {sum(v.size for v in init_params().values())}")
    print(f"  held-out prompts         {len(HELD_PAYLOADS)}, none seen in training")
    if RESULTS is None:
        print("  the seven fine-tunes     not run yet: they need exercises 1 to 3")
    else:
        print(f"  clean padded run         {RESULTS['clean, padded']['behaviour']['exact']} "
              f"exact of {len(HELD_PAYLOADS)}")
        print(f"  clean packed run         {RESULTS['clean, packed']['behaviour']['exact']} "
              f"exact of {len(HELD_PAYLOADS)}")
    print("  every number above was computed by the code you just ran")

In [ ]:
# Your progress board: one line per exercise, from the checks you have run, then the count.
# The eight exercises it lists are the ones `_EXERCISES` names in the setup cell.
_MARKS = {"passed": "✅ passed     ", "failed": "❌ failed     ", "not started": "⏳ not started"}


def _progress_board() -> None:
    """One line per exercise, from what the checks recorded, then how many are complete."""
    print("\nyour progress")
    done, next_up = 0, None
    for label, function, where in _EXERCISES:
        status = _STATUS.get(label, "not started")
        done += status == "passed"
        waiting = _WAITING_ON.get(label)
        note = (f"   (waiting on `{waiting}`)"
                if status == "not started" and waiting not in (None, "", function) else "")
        print(f"  {_MARKS[status]}  {label} · {function}{note}")
        if status != "passed" and next_up is None:
            next_up = (label, function, where)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    if next_up:
        print(f"next: {next_up[0]} — `{next_up[1]}`, {next_up[2]}. Its cell has hints if you "
              "are stuck.")


def _finish() -> None:
    """The board, then the verdict. A stub you have not reached yet is not a failure; a check
    that ran and came back wrong is. In a script or under CI that ends the run non-zero, so a
    green exit code can never paper over it. In a notebook kernel it is a printed line."""
    _progress_board()
    failed = [label for label, status in _STATUS.items() if status == "failed"]
    if failed and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(failed))
    if failed:
        print("checks failed: " + ", ".join(failed) + " — each one printed its likely mistake "
              "where it ran.")


if __name__ == "__main__":
    _finish()

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.